# Learner

> fastai Learners for spectral models

In [ ]:
#| default_exp learner

In [ ]:
#| export
from fastcore.all import *
from fastai.learner import Learner
from fastai.torch_core import params
from fastai.losses import MSELossFlat
from fastai.metrics import R2Score

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
from fastcore.test import *
from fastai.callback.schedule import *
from soilspecdl.datasets import load_toy_kssl
from soilspecdl.dataloader import get_spectral_dls
from soilspecdl.models import SpectralCNN
from soilspecdl.transforms import StandardizeSpectrum

A spectral model has two parts. The `backbone` turns each spectrum into a vector of features. The `head` maps those features to the prediction. `SpectralCNN` and its configurations, such as `MirzaiCNN`, follow this layout.

Fine-tuning a pretrained backbone treats the two parts differently. The head is new and trains first, while the backbone stays fixed. The backbone then adapts with a lower learning rate. fastai does both through parameter groups: `Learner.freeze` freezes every group except the last, and a `slice` of learning rates spreads them across the groups, lowest first. fastai's default `Learner` puts all parameters in one group, so it can do neither.

In [ ]:
#| export
def spectral_splitter(
    m, # Model with `backbone` and `head` attributes
)->list: # Parameter groups: backbone, then head
    "Split a spectral model's parameters into backbone and head groups"
    return L(m.backbone, m.head).map(params)

For the default `SpectralCNN`, the head holds the 257 parameters of `Linear(256, 1)`. The backbone holds the rest:

In [ ]:
model = SpectralCNN()
groups = spectral_splitter(model)
n_params = [sum(p.numel() for p in g) for g in groups]
test_eq(n_params[1], 257)
n_params

[131600, 257]

`spectral_learner` builds a `Learner` with `spectral_splitter`. Its defaults suit regression: mean squared error as the loss and R² as the metric. Pass `loss_func` and `metrics` for other tasks, such as classification.

In [ ]:
#| export
@delegates(Learner.__init__)
def spectral_learner(
    dls, # DataLoaders, e.g. from `get_spectral_dls`
    model, # Model with `backbone` and `head`, e.g. `SpectralCNN()`
    loss_func=None, # Loss function; `MSELossFlat()` if None
    metrics=None, # Metrics; `R2Score()` if None
    splitter=spectral_splitter, # Function splitting the model's parameters into groups
    **kwargs
)->Learner:
    "`Learner` for a spectral model, with backbone and head parameter groups"
    return Learner(dls, model, loss_func=ifnone(loss_func, MSELossFlat()), metrics=ifnone(metrics, R2Score()),
        splitter=splitter, **kwargs)

The examples train on the toy dataset from `soilspecdl.datasets`, predicting exchangeable potassium (Kex) from 500 KSSL MIR spectra:

In [ ]:
toy = load_toy_kssl()
X, y, ids = toy.xy('k.ext_usda.a725_cmolc.kg')
dls, test_idx = get_spectral_dls(X, y.flatten(), toy.wavenumbers, batch_tfms=[StandardizeSpectrum()])

The optimizer has one parameter group for the backbone and one for the head:

In [ ]:
learn = spectral_learner(dls, SpectralCNN())
learn.create_opt()
test_eq(len(learn.opt.param_groups), 2)
len(learn.opt.param_groups)

2

`freeze` stops training of the backbone. Following fastai's default `train_bn=True`, the backbone's normalization layers still train, so the head and the `BatchNorm` weights are the only trainable parameters:

In [ ]:
def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

learn.freeze()
test_eq(n_trainable(learn.model.head), 257)
n_trainable(learn.model.backbone), n_trainable(learn.model.head)

(992, 257)

Train the head first, with the backbone frozen. One epoch at each stage shows the steps, not a trained model:

In [ ]:
learn.fit_one_cycle(1, lr_max=1e-3)

epoch     train_loss  valid_loss  r2_score  time    
0         0.692485    0.263522    -0.072925 00:00     


Then unfreeze, and train the whole model with a `slice` of learning rates. The backbone gets the lower rate and the head the higher one:

In [ ]:
learn.unfreeze()
learn.opt.set_hyper('lr', slice(1e-5, 1e-3))
lrs = [h['lr'] for h in learn.opt.hypers]
test_close(lrs, [1e-5, 1e-3])
lrs

[1e-05, 0.001]

In [ ]:
learn.fit_one_cycle(1, lr_max=slice(1e-5, 1e-3))

epoch     train_loss  valid_loss  r2_score  time    
0         0.664211    0.316793    -0.289819 00:00     
